# Phase III: Text Summarization Inference Pipeline

Welcome to the inference phase of our Text Summarization project! In this notebook, we will set up the "brain" of our system—a pre-trained Transformer model—and use it to generate concise summaries of long articles.

### What is an Inference Pipeline?
An **inference pipeline** is a sequence of steps that takes raw data (like a news article), processes it so a machine can understand it (tokenization), feeds it into a specialized AI model, and then converts the model's complex mathematical output back into human-readable text (a summary).

### Key Technologies Used:
- **Hugging Face Transformers**: A library providing state-of-the-art AI models.
- **BART (Bidirectional and Auto-Regressive Transformers)**: Our chosen model, specifically designed for summarizing text.
- **PyYaml**: To load our project settings from a configuration file.

## 1. Setup and Project Configuration

Before we start, we need to load our settings from `config.yaml`. This file tells us which model to use and what the maximum lengths for our articles and summaries should be.

In [1]:
import yaml
import os
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load configuration
config_path = os.path.join("..", "config.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

model_name = config['model']['name']
max_input = config['model']['max_input_length']
max_output = config['model']['max_output_length']

print(f"Using model: {model_name}")
print(f"Max input length: {max_input} tokens")
print(f"Max output length: {max_output} tokens")

c:\Users\My Device\Desktop\Text Summarization Using Pre-trained Models\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using model: facebook/bart-base
Max input length: 1024 tokens
Max output length: 128 tokens


## 2. Loading the Model and Tokenizer

AI models don't read text like we do; they process numbers. 
- The **Tokenizer** breaks text into smaller chunks called "tokens" and maps them to numbers.
- The **Model** (BART) takes those numbers and predicts the next most likely words for our summary.

In [2]:
print("Loading tokenizer and model... This may take a moment depending on your internet connection.")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f"Model loaded successfully on: {device}")

Loading tokenizer and model... This may take a moment depending on your internet connection.


c:\Users\My Device\Desktop\Text Summarization Using Pre-trained Models\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\My Device\.cache\huggingface\hub\models--facebook--bart-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 259/259 [00:01<00:00, 152.22it/s, M

Model loaded successfully on: cpu


## 3. Creating the Summarization Pipeline

Hugging Face provides a "pipeline" object that simplifies the entire process. It handles tokenization, model prediction, and decoding automatically.

In [3]:
summarizer = pipeline(
    "summarization",
    model=model,
    tokenizer=tokenizer,
    device=0 if device == "cuda" else -1
)

print("Inference pipeline is ready!")

KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

## 4. Testing the System (Example Inference)

Now, let's feed a long news snippet into our system and see if it can capture the main points.

In [ ]:
sample_text = """
The Hubble Space Telescope has captured a stunning new image of a distant galaxy cluster, 
revealing thousands of stars and planetary systems in unprecedented detail. Scientists at NASA say 
this discovery could help us understand the early formation of the universe. The cluster, located 
billions of light-years away, shows signs of gravitational lensing, where the intense gravity of 
the cluster bends the light of even more distant objects behind it. This effect acts like a cosmic 
magnifying glass reaching back to the dawn of time.
"""

print("Generating summary...")

summary = summarizer(
    sample_text,
    max_length=max_output,
    min_length=30,
    do_sample=False,  # Use beam search for more consistent results
    truncation=True
)

print("\n--- Original Text ---")
print(sample_text.strip())

print("\n--- Generated Summary ---")
print(summary[0]['summary_text'])


### Conclusion
We now have a working system that can take any text and shorten it while keeping the core meaning! In the next phase, we will evaluate how "good" these summaries are compared to human-written ones using a metric called ROUGE.